<a href="https://colab.research.google.com/github/fbeilstein/bioinformatics/blob/master/practice_01_fastp_mash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Environment Setup

Before analyzing the mystery samples, we must install three standard bioinformatics tools into the Colab environment:

* **SRA Toolkit**: Provides the `fasterq-dump` command required to download raw sequencing datasets directly from the NCBI public databases using their unique SRR accession numbers.
* **fastp**: A quality control tool that cleans the raw sequencing data by trimming machine adapters and filtering out low-quality or erroneous reads.
* **Mash**: A genomic distance estimator. It uses the MinHash algorithm to compress massive sequence files into small signatures called "sketches," allowing for rapid comparison against reference genomes to identify the unknown sample.

In [ ]:
import subprocess
from IPython.display import clear_output

# Chain commands with && to ensure it stops if any step fails
command = """
wget -q http://opengene.org/fastp/fastp -O fastp && \
chmod a+x ./fastp && \
apt-get -y -qq install mash sra-toolkit
"""

result = subprocess.run(command, shell=True)

if result.returncode == 0:
    clear_output()
    print("Installations successful.")
else:
    print("An error occurred during installation. Remove -q and -qq flags to debug.")

### Datasets Downloading

After download the tool will print few numbers:
- spots read: In NCBI terminology, a "spot" represents a single physical DNA fragment (or cluster) on the sequencing flow cell.
- reads read: Because you downloaded paired-end data using --split-files, the sequencing machine read both ends of every DNA fragment. Therefore, the total number of reads is exactly twice the number of spots.
- reads written: If coincides with reads read --- the tool successfully saved all extracted reads to your disk without data loss or corruption.

In [ ]:
import os

# Map SRR accessions to descriptive names and numeric IDs
sample_mapping = {
    # The Baseline Reference
    "SRR40792821": "Reference_Ecoli",         # Escherichia coli

    # The Demonstration Sample
    "SRR40792818": "Mystery_Sample_00",       # Escherichia coli

    # Unknown organisms
    "SRR40769135": "Mystery_Sample_01",
    "SRR33253212": "Mystery_Sample_02",
    "SRR40784285": "Mystery_Sample_03",
    "SRR40783298": "Mystery_Sample_04",
    "SRR40792383": "Mystery_Sample_05",
    "SRR40789920": "Mystery_Sample_06",
    "SRR40760636": "Mystery_Sample_07"
}

for srr_id, talking_name in sample_mapping.items():
  print(f"Downloading {talking_name}")

  # 1. Download the paired-end files using fasterq-dump
  !fasterq-dump {srr_id} --split-files --progress

  # 2. Rename the downloaded files to the talking name
  !mv "{srr_id}_1.fastq" "{talking_name}_1_raw.fastq"
  !mv "{srr_id}_2.fastq" "{talking_name}_2_raw.fastq"

  if os.path.exists(f"{talking_name}_1_raw.fastq") and os.path.exists(f"{talking_name}_2_raw.fastq"):
    clear_output(wait=True)
  else:
    print(f"Error: {talking_name} failed to process. Halting.")
    break

print("download complete")

The `_1` and `_2` files indicate that the sample was processed using **paired-end sequencing**, where the sequencing machine reads both ends of a single DNA fragment.

* **The `_1` file:** Contains all the forward reads.
* **The `_2` file:** Contains all the reverse reads.

These files are perfectly synchronized. The read at line 500 in the `_1` file corresponds to the exact same physical piece of DNA as the read at line 500 in the `_2` file. Downstream bioinformatics tools require both files simultaneously because knowing the approximate physical distance between the forward and reverse reads helps algorithms accurately align the sequences to a reference genome.

### **Part 1:** Quality Control with `fastp`

When a sequencer processes a DNA strand, it detects physical signals (such as fluorescent light emissions) to determine the nucleotide. Because the hardware and chemistry are not perfect, the machine assigns a confidence score, known as a Phred quality score ($Q$), to every single base it reads. The score is logarithmically related to the probability of an incorrect base call ($P$):
$$
Q = -10 \log_{10}(P)
$$
- Q20: $P = 0.01$. There is a 1 in 100 chance the machine called the wrong base. If the machine outputs an "A" with a Q20 score, there is a 99% probability the physical molecule is Adenine, and a 1% probability it is actually Cytosine, Guanine, or Thymine.
- Q30: $P = 0.001$. There is a 1 in 1,000 chance the machine called the wrong base (99.9% accuracy).

When fastp reports the percentage of "bases with Q20," it is calculating what fraction of the total dataset meets or exceeds this 99% confidence threshold. For example, if a dataset has 95% Q20 bases, it means 95% of all A, C, T, and G calls in the file have a 99% or higher probability of being correct.

When `fastp` processes the raw data, it prints a statistical summary of the cleaning process directly to the console:

* **Before filtering / After filtering:** Displays the total reads, total bases, GC content, and the percentage of bases with Q20 (99% accuracy) and Q30 (99.9% accuracy) scores. Comparing these two sections shows the improvement in dataset quality.
* **Filtering results:** Shows the exact number of reads discarded because they were too short, contained too many unknown (`N`) bases, or fell below the quality threshold.
* **Duplication rate:** Estimates the percentage of identical reads, which can indicate PCR amplification bias during sequencing preparation.
* **Insert size peak:** For paired-end data, it estimates the average physical length of the sequenced DNA fragments.
* **HTML Report:** The tool generates an interactive `.html` file containing visual graphs of quality drops across the read length and adapter sequences removed.


In [ ]:
!./fastp -i Reference_Ecoli_1_raw.fastq -I Reference_Ecoli_2_raw.fastq -o Reference_Ecoli_1.fastq -O Reference_Ecoli_2.fastq -h Reference_Ecoli_report.html
!./fastp -i Mystery_Sample_00_1_raw.fastq -I Mystery_Sample_00_2_raw.fastq -o Mystery_Sample_00_1.fastq -O Mystery_Sample_00_2.fastq -h Mystery_Sample_00_report.html
!./fastp -i Mystery_Sample_01_1_raw.fastq -I Mystery_Sample_01_2_raw.fastq -o Mystery_Sample_01_1.fastq -O Mystery_Sample_01_2.fastq -h Mystery_Sample_01_report.html

### **Part 2:** MinHash Sketching with mash

Exact alignment is computationally expensive. We reduce the reference genomes and our cleaned reads into compressed MinHash sketches ($k=21$) to rapidly estimate the Jaccard similarity and Mash distance.

In [ ]:
!mash sketch -k 21 -m 2 Reference_Ecoli_1.fastq

!mash sketch -k 21 -m 2 Mystery_Sample_00_1.fastq
!mash sketch -k 21 -m 2 Mystery_Sample_01_1.fastq

When you run Mash to create a "sketch" of your sequence data, it analyzes the unique sequences (k-mers) in the FASTQ file and outputs a few key statistics before saving the final file:

* **Estimated genome size:** This is the algorithm's calculation of how large the organism's entire DNA sequence is (in base pairs). It is printed in scientific notation. For example, `7.50175e+06` means approximately 7.5 million base pairs.
* **Estimated coverage:** This indicates the "depth" of your sequencing. A coverage of `47.13` means that, on average, every single letter in the organism's genome was read 47 times by the sequencing machine. Higher coverage means higher confidence that the data accurately represents the real organism and isn't just a sequencing error.
* **Writing to [filename].msh:** Instead of comparing gigabytes of raw FASTQ text, Mash compresses the unique genomic patterns into a tiny, highly efficient binary file called a "sketch" (ending in `.msh`). This allows us to compare the mystery samples against reference databases in milliseconds rather than hours.

### **Part 3:** Jaccard Distance with mash

In [ ]:

# Calculate distance between the read sketch and both references
!mash dist Reference_Ecoli_1.fastq.msh Mystery_Sample_00_1.fastq
!mash dist Reference_Ecoli_1.fastq.msh Mystery_Sample_01_1.fastq

When you compare two sketches using mash dist, it outputs a tab-separated table with five distinct columns:

* **Reference ID:** The first file compared (Reference_Ecoli_1.fastq).
* **Query ID:** The second file compared (Mystery_Sample_0X_1.fastq).
* **Mash Distance:** A value between 0 and 1 estimating the mutation rate between the two genomes. A distance of 0 means the sequences are identical. A distance closer to 1 means they are entirely unrelated. Generally, a distance below 0.05 indicates the samples belong to the same species.
* **P-value:** The probability that this level of similarity occurred purely by random chance. A value of 0 or a very small scientific number (like 8.13809e-10) indicates extremely high statistical confidence in the calculation.
* **Matching Hashes:** The ratio of shared k-mer signatures out of the total sketch size (default is 1000). For example, 216/1000 means 216 out of 1000 unique genomic signatures matched perfectly.

### **Homework:** Mystery Sample Classification

Use your bioinformatics pipeline to identify the biological origin of each mystery sample.

- Go to [SRA Database](https://www.ncbi.nlm.nih.gov/sra/), choose SRA in the left field and type organism in the right one.
Fill in the correct SRR Accession number and sample ID next to the corresponding organism.
| Organism | Identified SRR # / Sample ID |
| :--- | :--- |
| *Bacillus subtilis* | SRR? |
| Ebola virus | SRR? |
| *Listeria monocytogenes* | SRR? |
| *Pseudomonas aeruginosa* | SRR? |
| *Salmonella enterica* | SRR? |
| SARS-CoV-2 | SRR? |
| *Staphylococcus aureus* (MRSA) | SRR? |

- Modify the code above to download these samples.
- Write the code that will classify mystery samples.
- Optimize the code for disk space consumption.